In [2]:
import os
import sys
import random
import copy
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

# Add project root to path for src imports
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '../..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.utils import seed_everything, load_config
seed_everything(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device configured: {device} | Seed: {seed_val}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 29.7 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.0 MB/s eta 0:00:00
✅ Using device: cuda


In [3]:
# Initialize OntologyEngine using relative path to ontology RDF
from src.ontology_engine import OntologyEngine, VSFCOntologyEngine
from src.utils import load_config

config_path = os.path.join(repo_root, 'configs/vsfc_ekman.yaml')
config = load_config(config_path)
onto_cfg = config.get('ontology', {})

ontology_path = os.path.join(repo_root, onto_cfg.get('rdf_path', 'ontology/ekman_appraisal_ontology.rdf'))

ontengine = OntologyEngine(
    rdf_path=ontology_path,
    default_conf=onto_cfg.get('default_conf', 0.85),
    manual_boost=onto_cfg.get('manual_boost', 1.2),
    negation_attenuation=onto_cfg.get('negation_attenuation', 0.5),
    alpha_similarity=onto_cfg.get('alpha_similarity', 0.2),
    verbose=True,
)


🚀 Initializing OntologyEngine v12.2 for VSFC (Deep Inference) with /kaggle/input/kg-ekman6/ekman6_4_8.rdf...
📥 Loading KG for VSFC: /kaggle/input/kg-ekman6/ekman6_4_8.rdf
   ✅ Loaded 90420 triples.
   📚 Lexicon size: 2540 entries (v12.2 VSFC-Ready)


In [4]:
# ---------------------------------------------------------
# CELL 3: DATASET & DATALOADER
# ---------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, ontology_engine, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.ont_engine = ontology_engine
        self.max_len = max_len
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        raw_text = str(self.df.iloc[index]['Sentence'])
        label = self.df.iloc[index]['label_idx']
        
        encoding = self.tokenizer.encode_plus(
            raw_text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        # Gọi hàm getvector (đã sửa tên đúng)
        ont_vector = self.ont_engine.getvector(raw_text) 
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'ontology_features': torch.tensor(ont_vector, dtype=torch.float),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Load Data
try:
    base_path = "../../data/vsfc-ekman" # Check path
    df_train = pd.read_csv(f"{base_path}/train.csv")
    df_valid = pd.read_csv(f"{base_path}/dev.csv")
    df_test = pd.read_csv(f"{base_path}/test.csv")
    
    for df in [df_train, df_valid, df_test]:
        df.rename(columns={'Text': 'Sentence', 'Ekman_Label': 'Emotion'}, inplace=True)
        df.dropna(subset=['Sentence', 'Emotion'], inplace=True)
    
    print("✅ Loaded Datasets.")
except Exception as e:
    print(f"❌ Error loading data: {e}")

# Label Encoding
le = LabelEncoder()
all_labels = sorted(df_train['Emotion'].unique().astype(str))
le.fit(all_labels)
print(f"📋 Classes: {le.classes_}")

for df in [df_train, df_valid, df_test]:
    df['label_idx'] = le.transform(df['Emotion'].astype(str))

# Dataloaders
BATCH_SIZE = 32
train_loader = DataLoader(EmotionDataset(df_train, tokenizer, ontengine), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(EmotionDataset(df_valid, tokenizer, ontengine), batch_size=BATCH_SIZE)
test_loader = DataLoader(EmotionDataset(df_test, tokenizer, ontengine), batch_size=BATCH_SIZE)

# Class Weights
train_labels = df_train['label_idx'].tolist()
class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"⚖️ Weights: {weights_tensor}")

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Loaded Datasets.
📋 Classes: ['Anger' 'Disgust' 'Fear' 'Happiness' 'Neutral' 'Sadness' 'Surprise']
⚖️ Weights: tensor([ 0.7617,  2.6327, 41.8535,  0.2899,  3.5717,  0.6545, 38.8639],
       device='cuda:0')


In [5]:
# Import PhoBERT Fusion architecture from src.models
from src.models.phobert_fusion_models import PhoBERT_Fusion_V2

try:
    dummy_vec = ontengine.getvector('test')
    if isinstance(dummy_vec, tuple): dummy_vec = dummy_vec[0]
    ONT_INPUT_DIM = len(dummy_vec)
except:
    ONT_INPUT_DIM = 24
print(f'✅ Loaded PhoBERT_Fusion_V2 | Detected Ontology Dimension: {ONT_INPUT_DIM}')


In [6]:
# ---------------------------------------------------------
# CELL 5: TRAINING HELPER FUNCTIONS (EARLY STOPPING)
# ---------------------------------------------------------
from sklearn.metrics import accuracy_score, f1_score

def train_experiment(model, name, epochs=20, patience=3):
    print(f"\n🚀 Bắt đầu huấn luyện: {name}")
    print(f"   (Cấu hình: Max Epochs={epochs}, Patience={patience})")

    # Bọc DataParallel nếu GPU > 1
    if torch.cuda.device_count() > 1:
        print(f"⚡ Đang chạy trên {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
    
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)
    
    best_f1 = 0
    # Copy weights ban đầu
    if isinstance(model, nn.DataParallel):
        best_model_wts = copy.deepcopy(model.module.state_dict())
    else:
        best_model_wts = copy.deepcopy(model.state_dict())
        
    patience_counter = 0 
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            ont = batch['ontology_features'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids, mask, ont)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        # Đánh giá sau mỗi epoch
        val_acc, val_f1 = evaluate(model, valid_loader)
        avg_loss = total_loss / len(train_loader)
        
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f}", end=" ")
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            if isinstance(model, nn.DataParallel):
                best_model_wts = copy.deepcopy(model.module.state_dict())
            else:
                best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0 
            print("✅ New Best!")
        else:
            patience_counter += 1
            print(f"⚠️ No improve ({patience_counter}/{patience})")
            
        if patience_counter >= patience:
            print(f"\n🛑 Early Stopping triggered! (Không cải thiện sau {patience} epochs)")
            break
            
    print(f"🏆 Kết thúc {name}. Best Val F1: {best_f1:.4f}")
    
    # Load lại best weights trước khi trả về
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(best_model_wts)
    else:
        model.load_state_dict(best_model_wts)
        
    return model

def evaluate(model, loader):
    model.eval()
    preds = []
    truths = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            ont = batch['ontology_features'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, mask, ont)
            _, pred = torch.max(outputs, dim=1)
            
            preds.extend(pred.cpu().numpy())
            truths.extend(labels.cpu().numpy())
            
    return accuracy_score(truths, preds), f1_score(truths, preds, average='macro')

In [7]:
import pandas as pd

basepath = "../../data/vsfc-ekman"

dftrain = pd.read_csv(f"{basepath}/train.csv")
dfvalid = pd.read_csv(f"{basepath}/dev.csv")
dftest  = pd.read_csv(f"{basepath}/test.csv")

# chuẩn hoá tên cột theo notebook của bạn
for df in [dftrain, dfvalid, dftest]:
    df.rename(columns={"Text": "Sentence", "Ekman_Label": "Emotion"}, inplace=True)

# bỏ NA
for df in [dftrain, dfvalid, dftest]:
    df.dropna(subset=["Sentence", "Emotion"], inplace=True)

print(len(dftrain), len(dfvalid), len(dftest))
print(dftrain.columns)
print(dftrain.head(2))


11426 1583 3166
Index(['Sentence', 'Original_Sentiment', 'Emotion'], dtype='object')
                                         Sentence  Original_Sentiment  \
0                       slide giáo trình đầy đủ .                   2   
1  nhiệt tình giảng dạy , gần gũi với sinh viên .                   2   

     Emotion  
0  Happiness  
1  Happiness  


In [8]:
import numpy as np
import pandas as pd

def add_onto_stats(df, ontengine, text_col="Sentence", eps=1e-6):
    # trả về df mới có các cột thống kê ontology
    rows = []
    for text in df[text_col].astype(str).tolist():
        v = ontengine.getvector(text)
        av = np.abs(v)

        rows.append({
            "onto_any": float(av.sum() > eps),
            "onto_emotion": float(av[:7].sum() > eps),
            "onto_appraisal": float(av[7:16].sum() > eps),
            "onto_negcue": float(av[-2] > 0.5),          # vn[0]
            "onto_nnz": int((av > eps).sum()),
        })
    out = df.copy()
    out = pd.concat([out.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
    return out

def coverage_by_label(df, label_col="Emotion"):
    g = df.groupby(label_col, dropna=False)
    rep = g.agg(
        N=(label_col, "size"),
        coverage_any_pct=("onto_any", "mean"),
        coverage_emotion_pct=("onto_emotion", "mean"),
        coverage_appraisal_pct=("onto_appraisal", "mean"),
        negcue_rate_pct=("onto_negcue", "mean"),
        avg_nnz_dims=("onto_nnz", "mean"),
    ).reset_index()

    # đổi mean -> %
    for c in ["coverage_any_pct","coverage_emotion_pct","coverage_appraisal_pct","negcue_rate_pct"]:
        rep[c] = (rep[c] * 100).round(3)

    rep["avg_nnz_dims"] = rep["avg_nnz_dims"].round(3)

    # sort: lớp nào ít mẫu lên trước hoặc theo negcue tùy bạn
    return rep.sort_values(["N"], ascending=True)

# --- chạy cho từng split ---
dftrain_s = add_onto_stats(dftrain, ontengine)
dfvalid_s = add_onto_stats(dfvalid, ontengine)  # nếu bạn có dfvalid
dftest_s  = add_onto_stats(dftest,  ontengine)

train_report = coverage_by_label(dftrain_s, label_col="Emotion")
val_report   = coverage_by_label(dfvalid_s, label_col="Emotion")
test_report  = coverage_by_label(dftest_s,  label_col="Emotion")

print("TRAIN\n", train_report.to_string(index=False))
print("\nVAL\n", val_report.to_string(index=False))
print("\nTEST\n", test_report.to_string(index=False))


TRAIN
   Emotion    N  coverage_any_pct  coverage_emotion_pct  coverage_appraisal_pct  negcue_rate_pct  avg_nnz_dims
     Fear   39            74.359                74.359                     0.0           15.385         2.590
 Surprise   42            92.857                83.333                     0.0           26.190         3.071
  Neutral  457            47.484                40.481                     0.0           15.755         1.024
  Disgust  620            85.000                71.774                     0.0           38.871         2.335
    Anger 2143            80.308                77.042                     0.0           16.146         2.144
  Sadness 2494            81.155                71.411                     0.0           28.027         2.176
Happiness 5631            80.465                79.950                     0.0            2.859         1.912

VAL
   Emotion   N  coverage_any_pct  coverage_emotion_pct  coverage_appraisal_pct  negcue_rate_pct  avg_nnz_dim

In [9]:
# =========================================================
# CELL: STRESS TEST LOGIC (V7.2)
# =========================================================
test_cases = [
    # 1. TEST PHỦ ĐỊNH + SCOPE (Liệu 'không' có xuyên qua 'hề' để bắt 'thích'?)
    ("tôi không hề thích nó", "Negation Scope Check"),
    
    # 2. TEST ĐẢO NGHĨA (Liệu 'Hạnh phúc' + 'Không' có thành 'Buồn'?)
    ("cảm thấy không vui chút nào", "Flip Logic Check"),
    
    # 3. TEST TỪ ĐA NGHĨA (Liệu 'chả' trong 'bún chả' có bị bắt nhầm là 'không'?)
    ("bún chả rất ngon", "Ambiguity Filter Check (Chả)"),
    
    # 4. TEST TỪ ĐA NGHĨA 2 (Liệu 'hay' có bị bắt nhầm là Vui?)
    ("làm hay nghỉ", "Ambiguity Filter Check (Hay)"),
    
    # 5. TEST CỤM TỪ CỐ ĐỊNH (MWE)
    ("thất vọng tràn trề", "MWE Recognition")
]

print(f"{'='*20} BÁO CÁO KẾT QUẢ TEST {'='*20}")

for text, desc in test_cases:
    print(f"\n🔹 CASE: '{text}' ({desc})")
    vec, tokens, trace = ontengine.getvector(text, debug=True)
    
    # In ra trace để xem Engine bắt từ nào
    print(f"   ► Tokens: {tokens}")
    if trace:
        print(f"   ► Matched: {[t['phrase'] for t in trace]}")
        # Check xem có Negation không
        is_neg = any(t['data']['isnegation'] for t in trace)
        print(f"   ► Detected Negation: {is_neg}")
    else:
        print("   ► Matched: [NONE]")
        
    # In ra các chiều vector có giá trị cao nhất
    labels = ontengine.ALLEMOTIONS
    emotions_vals = vec[:7] # 7 chiều đầu là Emotion
    max_idx = np.argmax(emotions_vals)
    
    if emotions_vals[max_idx] > 0:
        print(f"   ► DOMINANT EMOTION: {labels[max_idx]} (Score: {emotions_vals[max_idx]:.4f})")
    else:
        print(f"   ► DOMINANT EMOTION: [Neutral/None]")

print("\n" + "="*60)

==================== BÁO CÁO KẾT QUẢ TEST ====================

🔹 CASE: 'tôi không hề thích nó' (Negation Scope Check)
   ► Tokens: ['tôi', 'không', 'hề', 'thích', 'nó']
   ► Matched: ['không', 'thích']
   ► Detected Negation: True
   ► DOMINANT EMOTION: Happiness (Score: 0.4510)

🔹 CASE: 'cảm thấy không vui chút nào' (Flip Logic Check)
   ► Tokens: ['cảm thấy', 'không', 'vui', 'chút', 'nào']
   ► Matched: ['không', 'vui']
   ► Detected Negation: True
   ► DOMINANT EMOTION: Happiness (Score: 0.4941)

🔹 CASE: 'bún chả rất ngon' (Ambiguity Filter Check (Chả))
   ► Tokens: ['bún chả', 'rất', 'ngon']
   ► Matched: ['rất']
   ► Detected Negation: False
   ► DOMINANT EMOTION: Happiness (Score: 0.4189)

🔹 CASE: 'làm hay nghỉ' (Ambiguity Filter Check (Hay))
   ► Tokens: ['làm', 'hay', 'nghỉ']
   ► Matched: ['nghỉ']
   ► Detected Negation: False
   ► DOMINANT EMOTION: Sadness (Score: 0.3888)

🔹 CASE: 'thất vọng tràn trề' (MWE Recognition)
   ► Tokens: ['thất vọng', 'tràn trề']
   ► Matched: ['t

In [10]:
# ---------------------------------------------------------
# CELL 6: MAIN EXPERIMENTS (5 CASES)
# ---------------------------------------------------------
NC = len(le.classes_)

# 1. Baseline
m1 = PhoBERT_Fusion_V2(NC, 'none')
m1 = train_experiment(m1, "Case 1: Baseline", epochs=15)


# 3. Raw + Gate
m3 = PhoBERT_Fusion_V2(NC, 'gate', ontology_dim=None)
m3 = train_experiment(m3, "Case 3: Raw (24d) + Gate", epochs=15)

2026-02-24 09:11:31.170782: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771924291.352570      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771924291.403052      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771924291.860127      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771924291.860168      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771924291.860171      55 computation_placer.cc:177] computation placer alr

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🚀 Bắt đầu huấn luyện: Case 1: Baseline
   (Cấu hình: Max Epochs=15, Patience=3)


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Epoch 1/15 | Loss: 1.4835 | Val F1: 0.4450 ✅ New Best!
Epoch 2/15 | Loss: 1.1045 | Val F1: 0.5059 ✅ New Best!
Epoch 3/15 | Loss: 0.8524 | Val F1: 0.5711 ✅ New Best!
Epoch 4/15 | Loss: 0.6371 | Val F1: 0.4710 ⚠️ No improve (1/3)
Epoch 5/15 | Loss: 0.5200 | Val F1: 0.6130 ✅ New Best!
Epoch 6/15 | Loss: 0.4163 | Val F1: 0.6501 ✅ New Best!
Epoch 7/15 | Loss: 0.3287 | Val F1: 0.6274 ⚠️ No improve (1/3)
Epoch 8/15 | Loss: 0.2745 | Val F1: 0.6317 ⚠️ No improve (2/3)
Epoch 9/15 | Loss: 0.2309 | Val F1: 0.5994 ⚠️ No improve (3/3)

🛑 Early Stopping triggered! (Không cải thiện sau 3 epochs)
🏆 Kết thúc Case 1: Baseline. Best Val F1: 0.6501


Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🚀 Bắt đầu huấn luyện: Case 3: Raw (24d) + Gate
   (Cấu hình: Max Epochs=15, Patience=3)
Epoch 1/15 | Loss: 1.4569 | Val F1: 0.4382 ✅ New Best!
Epoch 2/15 | Loss: 1.0988 | Val F1: 0.4619 ✅ New Best!
Epoch 3/15 | Loss: 0.8413 | Val F1: 0.5985 ✅ New Best!
Epoch 4/15 | Loss: 0.6434 | Val F1: 0.5795 ⚠️ No improve (1/3)
Epoch 5/15 | Loss: 0.5013 | Val F1: 0.6215 ✅ New Best!
Epoch 6/15 | Loss: 0.3933 | Val F1: 0.6123 ⚠️ No improve (1/3)
Epoch 7/15 | Loss: 0.3102 | Val F1: 0.6196 ⚠️ No improve (2/3)
Epoch 8/15 | Loss: 0.2781 | Val F1: 0.6224 ✅ New Best!
Epoch 9/15 | Loss: 0.2253 | Val F1: 0.6247 ✅ New Best!
Epoch 10/15 | Loss: 0.2103 | Val F1: 0.6106 ⚠️ No improve (1/3)
Epoch 11/15 | Loss: 0.1870 | Val F1: 0.6267 ✅ New Best!
Epoch 12/15 | Loss: 0.1458 | Val F1: 0.6119 ⚠️ No improve (1/3)
Epoch 13/15 | Loss: 0.1068 | Val F1: 0.6157 ⚠️ No improve (2/3)
Epoch 14/15 | Loss: 0.1203 | Val F1: 0.6257 ⚠️ No improve (3/3)

🛑 Early Stopping triggered! (Không cải thiện sau 3 epochs)
🏆 Kết thúc Case 3: R

In [11]:
import torch
import os
from IPython.display import FileLink

# Tạo thư mục đầu ra
output_dir = "/kaggle/working/vsfc_ekman_checkpoints"
os.makedirs(output_dir, exist_ok=True)

# 1. Lưu Baseline (m1)
if 'm1' in globals():
    m1_path = os.path.join(output_dir, "vsfc_ekman_baseline_m1.pth")
    torch.save(m1.state_dict(), m1_path)
    print(f"✅ Đã lưu Baseline tại: {m1_path}")
    display(FileLink(r'vsfc_ekman_checkpoints/vsfc_ekman_baseline_m1.pth'))

# 2. Lưu Raw+Concat (m2 - xAI)
if 'm2' in globals():
    m2_path = os.path.join(output_dir, "vsfc_ekman_raw_concat_m2.pth")
    torch.save(m2.state_dict(), m2_path)
    print(f"✅ Đã lưu m2 tại: {m2_path}")
    display(FileLink(r'vsfc_ekman_checkpoints/vsfc_ekman_raw_concat_m2.pth'))

✅ Đã lưu Baseline tại: /kaggle/working/vsfc_ekman_checkpoints/vsfc_ekman_baseline_m1.pth


/kaggle/working/vsfc_ekman_checkpoints/vsfc_ekman_baseline_m1.pth

In [15]:
if 'm3' in globals():
    m3_path = os.path.join(output_dir, "vsfc_ekman_raw_gate_m3.pth")
    torch.save(m3.state_dict(), m3_path)
    print(f"✅ Đã lưu m2 tại: {m3_path}")
    display(FileLink(r'vsfc_ekman_checkpoints/vsfc_ekman_raw_gate_m3.pth'))

✅ Đã lưu m2 tại: /kaggle/working/vsfc_ekman_checkpoints/vsfc_ekman_raw_gate_m3.pth


/kaggle/working/vsfc_ekman_checkpoints/vsfc_ekman_raw_gate_m3.pth

In [12]:
# ---------------------------------------------------------
# CELL 6: MAIN EXPERIMENTS (5 CASES)
# ---------------------------------------------------------
NC = len(le.classes_)

# 1. Baseline
m1 = PhoBERT_Fusion_V2(NC, 'none')
m1 = train_experiment(m1, "Case 1: Baseline", epochs=15)


Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🚀 Bắt đầu huấn luyện: Case 1: Baseline
   (Cấu hình: Max Epochs=15, Patience=3)


KeyboardInterrupt: 

In [ ]:
# =========================================================
# CELL: ÁP DỤNG RULE-BASED TRÊN MODEL M1 (ĐÃ TRAIN)
# =========================================================
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm.auto import tqdm

# 1. Kiểm tra môi trường
if 'm1' not in globals():
    raise Exception("⚠️ Biến 'm1' chưa được định nghĩa! Vui lòng chạy cell train model m1 trước.")
if 'df_test' not in globals():
    raise Exception("⚠️ Biến 'df_test' chưa có. Vui lòng load dữ liệu trước.")

# 2. Định nghĩa bộ lý luận (Reasoner)
class NeuroSymbolicReasoner_M1:
    def __init__(self, model, engine, tokenizer, device, le_classes,
                 ont_threshold=0.6, model_conf_limit=0.9):
        self.model = model
        self.engine = engine
        self.tokenizer = tokenizer
        self.device = device
        self.classes = le_classes # ['Anger', 'Disgust', ..., 'Surprise']
        
        # Mapping tên nhãn -> index
        self.cls_to_idx = {name: i for i, name in enumerate(le_classes)}
        
        self.ont_threshold = ont_threshold      
        self.model_conf_limit = model_conf_limit 

    def infer(self, text):
        # --- BƯỚC 1: Neural Prediction (Dùng m1) ---
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding='max_length')
        input_ids = inputs['input_ids'].to(self.device)
        attention_mask = inputs['attention_mask'].to(self.device)
        
        # Lấy vector Ontology (cho model input - dù m1 fusion='none' vẫn cần tham số này nếu class định nghĩa vậy)
        vec, _, _ = self.engine.getvector(text, debug=True)
        ont_tensor = torch.tensor(np.array([vec]), dtype=torch.float).to(self.device)
        
        self.model.eval()
        with torch.no_grad():
            # Forward qua model m1
            outputs = self.model(input_ids, attention_mask, ont_tensor)
            probs = F.softmax(outputs, dim=1)
            
        pred_idx = torch.argmax(probs).item()
        confidence = probs[0][pred_idx].item()
        pred_label_str = self.classes[pred_idx]

        # --- BƯỚC 2: Symbolic Override (Luật Hậu xử lý) ---
        final_idx = pred_idx
        
        # RULE: Nếu Model đoán 'Neutral' VÀ độ tin cậy < 90%
        if pred_label_str == 'Neutral' and confidence < self.model_conf_limit:
            
            # Phân tích 7 điểm cảm xúc từ vector Ontology (7 index đầu)
            # Giả định thứ tự vector khớp với: Anger, Disgust, Fear, Happiness, Neutral, Sadness, Surprise
            ont_emotions = vec[:7] 
            max_ont_idx = np.argmax(ont_emotions)
            max_ont_val = ont_emotions[max_ont_idx]
            max_ont_label = self.engine.ALLEMOTIONS[max_ont_idx] 
            
            # Điều kiện Override:
            # 1. Điểm Ontology rất cao (> threshold)
            # 2. Nhãn Ontology KHÔNG phải Neutral
            if max_ont_val > self.ont_threshold and max_ont_label != 'Neutral':
                # Map tên nhãn Ontology sang index của Model
                if max_ont_label in self.cls_to_idx:
                    final_idx = self.cls_to_idx[max_ont_label]

        return pred_idx, final_idx

# 3. Chạy Đánh giá
print(f"🚀 Đang chạy Rule-based Post-processing trên model 'm1' ({len(df_test)} mẫu)...")

# Khởi tạo Reasoner với model m1
reasoner = NeuroSymbolicReasoner_M1(
    model=m1, # <--- DÙNG MODEL M1 CỦA BẠN
    engine=ontengine, 
    tokenizer=tokenizer, 
    device=device, 
    le_classes=le.classes_,
    ont_threshold=0.6 # Bạn có thể chỉnh 0.55 - 0.7
)

y_true = df_test['label_idx'].tolist()
y_base = []
y_rule = []

# Vòng lặp
for txt in tqdm(df_test['Sentence']):
    p_b, p_r = reasoner.infer(txt)
    y_base.append(p_b)
    y_rule.append(p_r)

# 4. Tính toán Metrics
acc_base = accuracy_score(y_true, y_base)
f1_base = f1_score(y_true, y_base, average='macro')

acc_rule = accuracy_score(y_true, y_rule)
f1_rule = f1_score(y_true, y_rule, average='macro')

# 5. Xuất Báo cáo
results_df = pd.DataFrame({
    "Method": ["1. Baseline (m1)", "2. m1 + Rules"],
    "Accuracy": [acc_base, acc_rule],
    "F1-Macro": [f1_base, f1_rule],
    "Diff (F1)": ["-", f"{f1_rule - f1_base:+.4f}"]
})

print("\n" + "="*50)
print("🏆 KẾT QUẢ SO SÁNH (M1 vs M1+RULES)")
print("="*50)
print(results_df.round(4).to_markdown(index=False))

print("\n📊 Chi tiết (m1 + Rules):")
print(classification_report(y_true, y_rule, target_names=le.classes_, digits=4))

# Đếm số ca sửa đổi
changes = sum(1 for a, b in zip(y_base, y_rule) if a != b)
print(f"\n🔄 Tổng số mẫu được luật can thiệp: {changes}/{len(df_test)} ({changes/len(df_test)*100:.2f}%)")

In [ ]:
# 2. Raw + Concat (Best xAI)
m2 = PhoBERT_Fusion_V2(NC, 'concat', ontology_dim=None)
m2 = train_experiment(m2, "Case 2: Raw (24d) + Concat", epochs=15)

# 3. Raw + Gate
m3 = PhoBERT_Fusion_V2(NC, 'gate', ontology_dim=None)
m3 = train_experiment(m3, "Case 3: Raw (24d) + Gate", epochs=15)



In [ ]:
# 4. Dense (64d) + Concat (Trade-off xAI for Nonlinearity)
m4 = PhoBERT_Fusion_V2(NC, 'concat', ontology_dim=64)
m4 = train_experiment(m4, "Case 4: Dense (64d) + Concat", epochs=15)



In [ ]:
# 5. Dense (64d) + Gate
m5 = PhoBERT_Fusion_V2(NC, 'gate', ontology_dim=64)
m5 = train_experiment(m5, "Case 5: Dense (64d) + Gate", epochs=15)

In [ ]:
# ---------------------------------------------------------
# CELL 7: FINAL REPORT (5 MODELS)
# ---------------------------------------------------------
def gp(model, loader):
    model.eval()
    p, t = [], []
    with torch.no_grad():
        for b in loader:
            ids, m, o, l = [b[k].to(device) for k in ['input_ids','attention_mask','ontology_features','labels']]
            _, pr = torch.max(model(ids, m, o), 1)
            p.extend(pr.cpu().numpy()); t.extend(l.cpu().numpy())
    return t, p

y_true, p1 = gp(m1, test_loader)
_, p2 = gp(m2, test_loader)
_, p3 = gp(m3, test_loader)
_, p4 = gp(m4, test_loader)
_, p5 = gp(m5, test_loader)

res = {
    "Model": ["Baseline", "Raw+Concat (xAI)", "Raw+Gate", "Dense(64)+Concat", "Dense(64)+Gate"],
    "Ontology Dim": ["-", "24 (Raw)", "24 (Raw)", "64 (Dense)", "64 (Dense)"],
    "F1-Macro": [f1_score(y_true, p, average='macro') for p in [p1, p2, p3, p4, p5]],
    "Accuracy": [accuracy_score(y_true, p) for p in [p1, p2, p3, p4, p5]]
}

print("\n" + "="*50)
print("🏆 FINAL THESIS RESULTS")
print("="*50)
print(pd.DataFrame(res).round(4).to_markdown(index=False))

print("\n" + "="*50)
print("📑 DETAILED REPORT (CASE 5: DENSE + GATE)")
print("="*50)
print(classification_report(y_true, p5, target_names=le.classes_, digits=4))

In [ ]:
# ---------------------------------------------------------\
# CELL: THỰC NGHIỆM CHIỀU SÂU (DEEP ONTOLOGY PROJECTION)
# Architecture: Input(24) -> Dense(64) -> ReLU -> Dense(768) -> Fusion
# ---------------------------------------------------------\
import torch
import torch.nn as nn
from transformers import AutoModel

# --- [FIX LỖI] ĐỊNH NGHĨA NUM_CLASSES ---
try:
    # Cố gắng lấy từ LabelEncoder nếu đã chạy cell data
    NUM_CLASSES = len(le.classes_)
    print(f"✅ Đã tìm thấy {NUM_CLASSES} nhãn từ LabelEncoder.")
except NameError:
    # Fallback nếu chưa có biến le
    NUM_CLASSES = 7
    print("⚠️ Không tìm thấy biến 'le', sử dụng mặc định NUM_CLASSES = 7 (VSFC Standard).")

# 1. Định nghĩa lại Model Deep Ontology
from src.models.phobert_fusion_models import PhoBERT_DeepOntology

# ---------------------------------------------------------
# CHẠY THỰC NGHIỆM
# ---------------------------------------------------------

# Lấy kích thước vector thực tế từ engine (thường là 24)
ONT_DIM = 24 

# === CASE A: DEEP PROJECTION + CONCAT ===
print("\n" + "="*40)
print("🧪 CASE A: Deep Ontology (24->64->768) + CONCAT")
print("="*40)
model_deep_concat = PhoBERT_DeepOntology(
    n_classes=NUM_CLASSES, 
    fusion_type='concat', 
    ont_input_dim=ONT_DIM, 
    hidden_dim=64
)
# In cấu trúc để kiểm tra
print(model_deep_concat.ont_pipeline) 

model_deep_concat = train_experiment(
    model_deep_concat, 
    "Deep Ont (64d) + Concat", 
    epochs=15, 
    patience=4
)
acc_a, f1_a = evaluate(model_deep_concat, test_loader)


# === CASE B: DEEP PROJECTION + GATE (PROPOSED) ===
print("\n" + "="*40)
print("🧪 CASE B: Deep Ontology (24->64->768) + GATE")
print("="*40)
model_deep_gate = PhoBERT_DeepOntology(
    n_classes=NUM_CLASSES, 
    fusion_type='gate', 
    ont_input_dim=ONT_DIM, 
    hidden_dim=64
)

model_deep_gate = train_experiment(
    model_deep_gate, 
    "Deep Ont (64d) + Gate", 
    epochs=15, 
    patience=4
)
acc_b, f1_b = evaluate(model_deep_gate, test_loader)

# === TỔNG HỢP KẾT QUẢ MỚI ===
print(f"\n📊 KẾT QUẢ KIẾN TRÚC 24->64->768:")
print(f"1. Concat: F1-Macro = {f1_a:.4f} | Acc = {acc_a:.4f}")
print(f"2. Gate  : F1-Macro = {f1_b:.4f} | Acc = {acc_b:.4f}")

In [ ]:
from src.models.phobert_fusion_models import PhoBERT_WideProjection


In [ ]:
# === CASE C: WIDE PROJECTION (768d) + CONCAT ===
print("\n" + "="*40)
print("🧪 CASE C: Wide Projection (24->768->768) + CONCAT")
print("="*40)
model_wide_concat = PhoBERT_WideProjection(n_classes=NUM_CLASSES, fusion_type='concat')

model_wide_concat = train_experiment(
    model_wide_concat, 
    "Wide Ont (768d) + Concat", 
    epochs=15, 
    patience=4
)
acc_c, f1_c = evaluate(model_wide_concat, test_loader)

# === CASE D: WIDE PROJECTION (768d) + GATE ===
print("\n" + "="*40)
print("🧪 CASE D: Wide Projection (24->768->768) + GATE")
print("="*40)
model_wide_gate = PhoBERT_WideProjection(n_classes=NUM_CLASSES, fusion_type='gate')

model_wide_gate = train_experiment(
    model_wide_gate, 
    "Wide Ont (768d) + Gate", 
    epochs=15, 
    patience=4
)
acc_d, f1_d = evaluate(model_wide_gate, test_loader)

# --- TỔNG KẾT ---
print(f"\n📊 KẾT QUẢ KIẾN TRÚC WIDE PROJECTION (24->768->768):")
print(f"1. Wide + Concat: F1-Macro = {f1_c:.4f} | Acc = {acc_c:.4f}")
print(f"2. Wide + Gate  : F1-Macro = {f1_d:.4f} | Acc = {acc_d:.4f}")

In [ ]:
# =========================================================
# CELL: ÁP DỤNG RULE-BASED TRÊN MODEL M1 (ĐÃ TRAIN)
# =========================================================
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm.auto import tqdm

# 1. Kiểm tra môi trường
if 'm1' not in globals():
    raise Exception("⚠️ Biến 'm1' chưa được định nghĩa! Vui lòng chạy cell train model m1 trước.")
if 'df_test' not in globals():
    raise Exception("⚠️ Biến 'df_test' chưa có. Vui lòng load dữ liệu trước.")

# 2. Định nghĩa bộ lý luận (Reasoner)
class NeuroSymbolicReasoner_M1:
    def __init__(self, model, engine, tokenizer, device, le_classes,
                 ont_threshold=0.6, model_conf_limit=0.9):
        self.model = model
        self.engine = engine
        self.tokenizer = tokenizer
        self.device = device
        self.classes = le_classes # ['Anger', 'Disgust', ..., 'Surprise']
        
        # Mapping tên nhãn -> index
        self.cls_to_idx = {name: i for i, name in enumerate(le_classes)}
        
        self.ont_threshold = ont_threshold      
        self.model_conf_limit = model_conf_limit 

    def infer(self, text):
        # --- BƯỚC 1: Neural Prediction (Dùng m1) ---
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding='max_length')
        input_ids = inputs['input_ids'].to(self.device)
        attention_mask = inputs['attention_mask'].to(self.device)
        
        # Lấy vector Ontology (cho model input - dù m1 fusion='none' vẫn cần tham số này nếu class định nghĩa vậy)
        vec, _, _ = self.engine.getvector(text, debug=True)
        ont_tensor = torch.tensor(np.array([vec]), dtype=torch.float).to(self.device)
        
        self.model.eval()
        with torch.no_grad():
            # Forward qua model m1
            outputs = self.model(input_ids, attention_mask, ont_tensor)
            probs = F.softmax(outputs, dim=1)
            
        pred_idx = torch.argmax(probs).item()
        confidence = probs[0][pred_idx].item()
        pred_label_str = self.classes[pred_idx]

        # --- BƯỚC 2: Symbolic Override (Luật Hậu xử lý) ---
        final_idx = pred_idx
        
        # RULE: Nếu Model đoán 'Neutral' VÀ độ tin cậy < 90%
        if pred_label_str == 'Neutral' and confidence < self.model_conf_limit:
            
            # Phân tích 7 điểm cảm xúc từ vector Ontology (7 index đầu)
            # Giả định thứ tự vector khớp với: Anger, Disgust, Fear, Happiness, Neutral, Sadness, Surprise
            ont_emotions = vec[:7] 
            max_ont_idx = np.argmax(ont_emotions)
            max_ont_val = ont_emotions[max_ont_idx]
            max_ont_label = self.engine.ALLEMOTIONS[max_ont_idx] 
            
            # Điều kiện Override:
            # 1. Điểm Ontology rất cao (> threshold)
            # 2. Nhãn Ontology KHÔNG phải Neutral
            if max_ont_val > self.ont_threshold and max_ont_label != 'Neutral':
                # Map tên nhãn Ontology sang index của Model
                if max_ont_label in self.cls_to_idx:
                    final_idx = self.cls_to_idx[max_ont_label]

        return pred_idx, final_idx

# 3. Chạy Đánh giá
print(f"🚀 Đang chạy Rule-based Post-processing trên model 'm1' ({len(df_test)} mẫu)...")

# Khởi tạo Reasoner với model m1
reasoner = NeuroSymbolicReasoner_M1(
    model=m1, # <--- DÙNG MODEL M1 CỦA BẠN
    engine=ontengine, 
    tokenizer=tokenizer, 
    device=device, 
    le_classes=le.classes_,
    ont_threshold=0.6 # Bạn có thể chỉnh 0.55 - 0.7
)

y_true = df_test['label_idx'].tolist()
y_base = []
y_rule = []

# Vòng lặp
for txt in tqdm(df_test['Sentence']):
    p_b, p_r = reasoner.infer(txt)
    y_base.append(p_b)
    y_rule.append(p_r)

# 4. Tính toán Metrics
acc_base = accuracy_score(y_true, y_base)
f1_base = f1_score(y_true, y_base, average='macro')

acc_rule = accuracy_score(y_true, y_rule)
f1_rule = f1_score(y_true, y_rule, average='macro')

# 5. Xuất Báo cáo
results_df = pd.DataFrame({
    "Method": ["1. Baseline (m1)", "2. m1 + Rules"],
    "Accuracy": [acc_base, acc_rule],
    "F1-Macro": [f1_base, f1_rule],
    "Diff (F1)": ["-", f"{f1_rule - f1_base:+.4f}"]
})

print("\n" + "="*50)
print("🏆 KẾT QUẢ SO SÁNH (M1 vs M1+RULES)")
print("="*50)
print(results_df.round(4).to_markdown(index=False))

print("\n📊 Chi tiết (m1 + Rules):")
print(classification_report(y_true, y_rule, target_names=le.classes_, digits=4))

# Đếm số ca sửa đổi
changes = sum(1 for a, b in zip(y_base, y_rule) if a != b)
print(f"\n🔄 Tổng số mẫu được luật can thiệp: {changes}/{len(df_test)} ({changes/len(df_test)*100:.2f}%)")